## Building a Neural Network with Hidden Layers from scratch using Numpy and scipy

This neural network will have:
1. Input Layer with 3 neurons
2. Hidden Layer with 2 neurons
3. Output layer with 1 neuron

Properties of the neural network: 
- Hidden layers will have sigmoid and relu functions as the activations with backpropagation. 
- Loss function cross entropy loss for classification and MSELoss for regression will also be coded from scratch.
- Optimizers optim.SGD or the Stochastic Gradient Descent and optim.Adam or the Adam will be coded from scratch as well. 
- Finally lets compare this with torch for time taken
- Lastly, converting them into ONNX by hand using protobuf will also be properly done so that they can be displayed in netron.app

### We will follow an OOP structure with maintaining them as different submodules for different classes. First, lets start with working on a Neuron and training it

In [ ]:
from typing import Optional, List, Any, Tuple, Union
import torch
import pandas as pd
import numpy as np
import scipy.linalg as linal

In [ ]:
class MatMulError(Exception):
    def __init__(self,message):
        self.message = message
        super().__init__(message)
    def __str__(self):
        return f"{self.message} has been raised when performing Matrix Multiplication"
    
class DimensionMisMatchError(Exception):
    def __init__(self,message):
        self.message = message 
        super().__init__(message)
    def __str__(self):
        return f"{self.message} has been raised when checking for dimension match"

In [ ]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_dash(x):
    return sigmoid(x)*(1-sigmoid(x))

def cross_entropy_loss(y_true, y_pred, reduction: str = 'mean'):
    """ 
    Produces the mean or sum of the Cross Entropy Loss over the batch
    """
    # naive logic
    # true_idx_1 = np.where(y_true==0)[0]
    # true_idx_2 = np.where(y_true==1)[0]
    # loss = -(y_true[true_idx_2]*np.log(y_pred[true_idx_2])).sum()-((1-y_true[true_idx_1])*np.log(1 - y_pred[true_idx_1])).sum() 

    y_pred = np.clip(y_pred, 1e-15, 1-1e-15) # to keep the array from having 0 or 1 values so that the log might explode or cause NaN values
    loss = -np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    if reduction == 'mean':
        return loss.sum()/y_true.shape[0]
    else:
        return loss.sum()

def cross_entropy_loss_derivative(y_true, y_pred):
    return (y_pred-y_true)

def accuracy(y_true, y_pred):
    return float((y_true.reshape(-1,1)==y_pred.reshape(-1,1)).sum()/y_pred.shape[0])*100

In [ ]:
class Neuron:
    def __init__(self, in_features:int):
        self.in_f = in_features 
        self.w = np.random.randn(self.in_f,1) # m x 1 weight matrix for m features and 1 output value
        self.b = np.random.random() # 1 x 1 bias matrix for the neuron

    def fit(self,X: Union[np.array, torch.tensor, pd.DataFrame], y: Union[np.array, torch.tensor, pd.DataFrame], 
            n_epochs:int = 100, lr:float = 1e-4, verbose: bool = False, 
            frequency: int = 5):
        """ 
        Function to fit the given dataset with the Neuron's Sigmoid function and train according to the given hyperparameters.
        Uses Standard Gradient Descent.
        """
        Y = y.reshape(-1,1) # to make the dimensions n x 1 as a 2D matrix instead of a 1D array
        for epoch in range(n_epochs):
            y_pred = self.predict_proba(X)
            loss = cross_entropy_loss(Y,y_pred)
            loss_der = cross_entropy_loss_derivative(Y,y_pred)
            w = self.w - lr*(X.T)@(loss_der)
            b = self.b - lr*np.sum(loss_der)/X.shape[0]
            self.w = w
            self.b = b
            if epoch%frequency == 0 or verbose:
                print(f"Epoch: {epoch}, loss: {loss}")

    def predict(self, X: Union[np.array, torch.tensor, pd.DataFrame]):
        if len(X.shape) > 2:
            raise DimensionMisMatchError(f"X is 3 dimensional with {X.shape}, expected 2 dimensional.")
        if not self.w.shape[0] in X.shape:
            raise MatMulError(f"Shape mismatch: Matrices with sizes {self.w.shape} cannot be multiplied with {X.shape}") 
        try:
            if self.w.shape[0] == X.shape[1]:
                return np.array(sigmoid(X@self.w+self.b) >=0.5, dtype=float) # returns n x 1 array
            elif self.w.shape[0] == X.shape[0]:
                return np.array(sigmoid(X.T@self.w+self.b) >=0.5, dtype=float) # return n x 1 array
        except Exception as e:
            print(f"Receiving error : {e}")
            return None
    
    def predict_proba(self, X: Union[np.array, torch.tensor, pd.DataFrame]):
        if len(X.shape) > 2:
            raise DimensionMisMatchError(f"X is 3 dimensional with {X.shape}, expected 2 dimensional.")
        if not self.w.shape[0] in X.shape:
            raise MatMulError(f"Shape mismatch: Matrices with sizes {self.w.shape} cannot be multiplied with {X.shape}") 
        try:
            if self.w.shape[0] == X.shape[1]:
                return sigmoid(X@self.w+self.b) # returns n x 1 array
            elif self.w.shape[0] == X.shape[0]:
                return sigmoid(X.T@self.w+self.b) # return n x 1 array
        except Exception as e:
            print(f"Receiving error : {e}")
            return None

#### Lets generate a sample dataset with gaussian noise

In [ ]:
X = np.random.randn(1000,6)
y = np.array([0,1]*500)
np.random.shuffle(y)
neuron = Neuron(6)

In [ ]:
neuron.fit(X,y,100,0.001,False)

Epoch: 0, loss: 1.080401446258235
Epoch: 5, loss: 0.8382047823125
Epoch: 10, loss: 0.7887549435706037
Epoch: 15, loss: 0.7822426764168877
Epoch: 20, loss: 0.781283171878581
Epoch: 25, loss: 0.7809857973752983
Epoch: 30, loss: 0.7807675999891314
Epoch: 35, loss: 0.7805592252268183
Epoch: 40, loss: 0.7803523937792909
Epoch: 45, loss: 0.7801461150318794
Epoch: 50, loss: 0.7799402698727503
Epoch: 55, loss: 0.7797348434678629
Epoch: 60, loss: 0.7795298334608872
Epoch: 65, loss: 0.7793252389882067
Epoch: 70, loss: 0.7791210593648757
Epoch: 75, loss: 0.7789172939275397
Epoch: 80, loss: 0.7787139420156862
Epoch: 85, loss: 0.7785110029694136
Epoch: 90, loss: 0.7783084761291684
Epoch: 95, loss: 0.77810636083572


In [ ]:
accuracy(neuron.predict(X),y)

50.0

In [ ]:
X = np.random.randn(10000,8)
y = np.array((5*X**2+23*X+np.random.sample((X.shape))).sum(axis=1) > 38,dtype=float) 

In [ ]:
neuron = Neuron(8)
neuron.fit(X,y,1000,1e-5,False,frequency=50)

Epoch: 0, loss: 1.247322351491632
Epoch: 50, loss: 0.5481714269663633
Epoch: 100, loss: 0.3448428527089409
Epoch: 150, loss: 0.28754147400965846
Epoch: 200, loss: 0.2660560772933944
Epoch: 250, loss: 0.25491875505720074
Epoch: 300, loss: 0.24768367444782674
Epoch: 350, loss: 0.24236634705693072
Epoch: 400, loss: 0.23820165128325863
Epoch: 450, loss: 0.2348226065301374
Epoch: 500, loss: 0.2320195989804385
Epoch: 550, loss: 0.22965776433791088
Epoch: 600, loss: 0.22764355380639467
Epoch: 650, loss: 0.22590889352343538
Epoch: 700, loss: 0.2244026167410077
Epoch: 750, loss: 0.22308534493372909
Epoch: 800, loss: 0.22192620714061012
Epoch: 850, loss: 0.22090063171283164
Epoch: 900, loss: 0.21998880645900007
Epoch: 950, loss: 0.21917457578980276


In [ ]:
y_pred = neuron.predict(X)
accuracy(y, y_pred)

90.02

In [ ]:
(y.reshape(-1,1) != y_pred).sum()

np.int64(998)

### Autograd in numpy and python from scratch

In [58]:
class GraphPropagationError(Exception):
    def __init__(self,message: str=None):
        self.message=message
        super().__init__(message)


In [ ]:
from typing import Union, Self
import numpy as np

class Node:
    ## No parallel processing yet. Because we do not handle Atomic operations of any variables like counters and dynamic graph
    
    NAMEVARCOUNTER = 1
    NAMEFUNCTIONCOUNTER = 1
    Graph = []
    def  __init__(self, type:str):
        self.name = None
        self.type = type
        self.value = None
        self.gradient = None
        self.next = None
        self.prev = None
        self.gradbackprop = None
        Node.Graph.append(self)
    
    def __str__(self):
        return f"{self.name} {self.type} node"

    def __repr__(self):
        return str(self)
    
    def __value__(self):
        return self.value

    def is_leaf(self):
        return self.prev==None

    def is_root(self):
        return self.next==None

class Value(Node):
    def __init__(self,value: Union[int, float, np.array] = None, name:str=None):
        super().__init__("value")
        self.value = value if type(value)!=Node else value.value
        if name:self.name=name
        else:
            self.name=f"var_{Node.NAMEVARCOUNTER}"
            Node.NAMEVARCOUNTER+=1
        print(f"{self.name} variable has been created")
    
    def __add__(self,right: Union[Self, int, float,np.array]):
        add = Add(self,right)
        add.apply()
        self.next = add
        if isinstance(right,Value):right.next = add
        output = add.output
        add.next = output
        output.prev = add
        return output
    
    def __mul__(self,right: Union[Self, int, float,np.array]):
        mul = Multiply(self,right)
        mul.apply()
        output = mul.output
        self.next = mul
        mul.next = output
        output.prev = mul
        if isinstance(right,Value):right.next = mul
        return output
    
    def __rmul__(self,right:Union[Self, int, float,np.array]):
        output = self*right
        return output

    def __sub__(self,right: Union[Self, int, float,np.array]):
        sub = Subtract(self,right)
        sub.apply()
        output = sub.output
        self.next = sub
        if isinstance(right,Value):right.next = sub
        sub.next = output
        output.prev = sub
        return output
    
    def __pow__(self, right: int):
        pow = Pow(self,right)
        pow.apply()
        output = pow.output
        self.next = pow
        pow.next = output
        output.prev = pow
        return output

    def backward(self):
        if self.is_root():
            self.gradient = 1
        else:
            if self.next == None:
                raise GraphPropagationError("Not a root node, but encountered an empty next node. Formation error?")
            if self.next.type == 'function':
                self.gradient = self.gradbackprop
            elif self.next.type=='value':
                raise GraphPropagationError("Encountered a value node followed by a value node. Formation error?")
            else:
                raise GraphPropagationError("Encountered unknown error. ")
        
        if not self.is_leaf():
            self.prev.backward()

class Function(Node):
    def __init__(self,operation :str = None,name: str=None):
        super().__init__('function')
        self.left_operand = None
        self.right_operand = None
        self.operation = operation
        self.value = operation
        if name:self.name=name
        else:
            self.name=f"func_{Node.NAMEFUNCTIONCOUNTER}"
            Node.NAMEFUNCTIONCOUNTER+=1
        print(f"{self.name} function {self.operation} has been created")

class Add(Function):
    def __init__(self, left_operand: Union[Value, int, float,np.array], right_operand: Union[Value, int, float], output: Union[Value, int, float, np.array] = None):
        super().__init__('+')
        self.left_operand=left_operand
        self.right_operand=right_operand
        self.output = output if not isinstance(output,Value) else Value(output)
    
    def apply(self):
        if isinstance(self.right_operand,Value):
            right = self.right_operand.value
        else:
            right = self.right_operand
        self.output.value = (self.left_operand.value + right)
    
    def backward(self):
        self.left_operand.gradbackprop = 1*self.next.gradient
        self.left_operand.backward()

        if isinstance(self.right_operand,Value):
            self.right_operand.gradbackprop = 1*self.next.gradient
            self.right_operand.backward()
        

class Subtract(Function):
    def __init__(self, left_operand: Union[Value, int, float,np.array], right_operand: Union[Value, int, float], output: Union[Value, int, float, np.array] = None):
        super().__init__('-')
        self.left_operand=left_operand
        self.right_operand=right_operand
        self.output = output if not isinstance(output,Value) else Value(output)
    
    def apply(self):
        if isinstance(self.right_operand,Value):
            right = self.right_operand.value
        else:
            right = self.right_operand
        self.output.value = self.left_operand.value - right
    
    def backward(self):
        self.left_operand.gradbackprop = 1*self.next.gradient
        self.left_operand.backward()
        if isinstance(self.right_operand,Value):
            self.right_operand.gradbackprop = -1*self.next.gradient
            self.right_operand.backward()

class Multiply(Function):
    def __init__(self, left_operand: Union[Value, int, float,np.array], right_operand: Union[Value,int, float], output: Union[Value, int, float, np.array] = None):
        super().__init__('*')
        self.left_operand=left_operand
        self.right_operand=right_operand
        self.output = output if not isinstance(output,Value) else Value(output)
    
    def apply(self):
        if isinstance(self.right_operand,Value):
            right = self.right_operand.value
        else:
            right = self.right_operand
        self.output.value = self.left_operand.value*right
    
    def backward(self):
        if type(self.right_operand)!=Value:
            right = self.right_operand
        else:
            right = self.right_operand.value
            self.right_operand.gradbackprop = self.left_operand.value*self.next.gradient
            self.right_operand.backward()
        
        self.left_operand.gradbackprop = right*self.next.gradient
        self.left_operand.backward()
        

class Pow(Function):
    def __init__(self, left_operand: Union[Value, int, float,np.array], right_operand: int, output: Union[Value, int, float, np.array] = None):
        super().__init__('**')
        self.left_operand=left_operand
        self.right_operand=right_operand
        self.output = output if not isinstance(output,Value) else Value(output)
    
    def apply(self):
        self.output.value = self.left_operand.value ** self.right_operand
    
    def backward(self):
        right = self.right_operand
        self.left_operand.gradbackprop = right*(self.left_operand.value ** (right-1))*self.next.gradient
        self.left_operand.backward()

class Sigmoid(Function):
    def __init__(self,left_operand: Union[Value,int,float,np.array],output: Union[Value, int, float, np.array]=None):
        super().__init__("sigmoid")
        self.left_operand = left_operand
        self.output = output if not isinstance(output,Value) else Value(output)
    
    def apply(self):
        if isinstance(self.left_operand,Value):value = self.left_operand.value
        else: value=self.left_operand
        self.output.value = 1/(1+np.exp(value))
    
    def backward(self):
        sigm = self.output.value
        if isinstance(self.left_operand,Value):
            self.left_operand.gradbackprop = sigm*(1-sigm)
            self.left_operand.backward()

In [177]:
A = Value(13)
B = Value(65)

var_1 variable has been created
var_2 variable has been created


In [178]:
C = A**3
D = C*4
E = B**2
F = E*2
G = D-F
G.backward()
# output = (value1**3)*4 - (value2**2)*2

func_1 function ** has been created
var_3 variable has been created
func_2 function * has been created
var_4 variable has been created
func_3 function ** has been created
var_5 variable has been created
func_4 function * has been created
var_6 variable has been created
func_5 function - has been created
var_7 variable has been created


In [158]:
A.gradient, B.gradient, C.gradient, D.gradient, E.gradient, F.gradient, G.gradient

(2028, -260, 4, 1, -2, -1, 1)

In [203]:
A = Value(4, 'A')
B = Value(3, 'B')
C = Value(2, 'C')

D = A*B*(C**4 + 10)

A variable has been created
B variable has been created
C variable has been created
func_1 function * has been created
var_1 variable has been created
func_2 function ** has been created
var_2 variable has been created
func_3 function + has been created
var_3 variable has been created
func_4 function * has been created
var_4 variable has been created


In [197]:
Node.Graph

[A value node,
 B value node,
 C value node,
 func_1 function node,
 var_1 value node,
 func_2 function node,
 var_2 value node,
 func_3 function node,
 var_3 value node,
 func_4 function node,
 var_4 value node]

In [204]:
D.backward()

In [205]:
A.gradient, B.gradient, C.gradient

(78, 104, 384)

In [206]:
for node in Node.Graph:
    print(node.name,node.type,node.next,node.gradient)

A value func_1 function node 78
B value func_1 function node 104
C value func_2 function node 384
func_1 function var_1 value node None
var_1 value func_4 function node 26
func_2 function var_2 value node None
var_2 value func_3 function node 12
func_3 function var_3 value node None
var_3 value func_4 function node 12
func_4 function var_4 value node None
var_4 value None 1


### Accessing from deeptorch folder

In [1]:
import os
os.getcwd()
os.chdir("..")

In [2]:
from deeptorch.autograph import *
from deeptorch.utils import *

A = Value(np.random.randn(4,5))

var_1 variable has been created


In [3]:
B = Sigmoid(A)

func_1 function sigmoid has been created
var_2 variable has been created


In [4]:
B.apply()
B.backward()

Existing sigmoid value:  [[0.42130758 0.82711408 0.78502439 0.2396289  0.38155841]
 [0.57399876 0.30302606 0.5126286  0.82220035 0.32588861]
 [0.62478264 0.33426762 0.50176442 0.3464837  0.69007523]
 [0.6269536  0.40405235 0.22727142 0.17793228 0.33684983]]


In [5]:
A.gradient

array([[0.2438075 , 0.14299638, 0.1687611 , 0.18220689, 0.23597159],
       [0.24452418, 0.21120127, 0.24984052, 0.14618693, 0.21968522],
       [0.23442929, 0.22253278, 0.24999689, 0.22643275, 0.21387141],
       [0.23388278, 0.24079405, 0.17561912, 0.14627238, 0.22338202]])

In [6]:
B.gradient

In [9]:
C = Value(0.5)
D = Sigmoid(C)

var_4 variable has been created
func_2 function sigmoid has been created
var_5 variable has been created


In [10]:
D.output.__value__()

np.float64(0.3775406687981454)

In [11]:
D.backward()

Existing sigmoid value:  0.3775406687981454


In [12]:
C.gradient

np.float64(0.2350037122015945)

In [1]:
import os
os.chdir("..")
from deeptorch.autograph import *
from deeptorch.utils import *

A = Value(np.random.randn(4,5))

var_1 variable has been created


In [3]:
B = Sigmoid()

func_1 function sigmoid has been created
var_2 variable has been created


In [4]:
B.apply(A)

In [5]:
B.next.backward()

In [6]:
B.gradient

In [7]:
A.gradient

array([[0.24873004, 0.23796117, 0.24296257, 0.24829315, 0.11136678],
       [0.20729691, 0.22113299, 0.22133339, 0.24465405, 0.19102937],
       [0.20724069, 0.20951174, 0.24289798, 0.20352305, 0.21464366],
       [0.14947626, 0.2490574 , 0.1714828 , 0.19969232, 0.24521596]])

In [2]:
Sigmoid(A.value).apply()*(1-Sigmoid(A.value).apply())

func_1 function sigmoid has been created
var_2 variable has been created
func_2 function sigmoid has been created
var_3 variable has been created
func_3 function - has been created
var_4 variable has been created
func_4 function * has been created
var_5 variable has been created


array([[-0.17902243, -0.17407795, -0.24792213, -0.23072778, -0.24994224],
       [-0.23425824, -0.22944905, -0.23772229, -0.23100099, -0.23090416],
       [-0.24946869, -0.24220543, -0.17676439, -0.16822784, -0.20463008],
       [-0.24936503, -0.23783911, -0.20427122, -0.24999104, -0.24325277]])

In [5]:
A.value is not None

True